# Estructura del notebook
1. Realizar benchmark de generacion de trazas no procesadas (timestamp, hex data separadas por espacios)
2. 

## Generador de benchmark

In [1]:
import pandas as pd
import modbus_parser
import time
import glob
from datetime import datetime

In [ ]:

# Buscar y unir todos los archivos crudos en el directorio de entrada
input_files = glob.glob("./data/input/raw_*.csv")
print(f"Cargando datos en memoria desde {len(input_files)} archivos...\n")

# Concatenar todos los archivos encontrados en un solo DataFrame
df_raw = pd.concat((pd.read_csv(f) for f in input_files), ignore_index=True)
df_raw = df_raw.dropna(subset=['hex_data', 'timestamp'])

fecha_actual = datetime.now().strftime("%Y%m%d_%H%M%S")

In [ ]:
# IMPORTANTE LEER
# Si es verdadero, genera el archivo procesado final en output_file
PROCESS_DATA = True
output_file = f"./data/output/processed_traces/pr_ts-{fecha_actual}.csv"

In [ ]:
# ---------------------------------------------------------
# Prueba 1: Secuencial (Bucle For en Python)
# ---------------------------------------------------------
print("Ejecutando método 1 (Secuencial)...")
start_seq = time.perf_counter()

resultados_validos = []
for index, row in df_raw.iterrows():
    timestamp = row['timestamp']
    hex_string = row['hex_data']

    try:
        raw_bytes = bytes.fromhex(hex_string.replace(" ", ""))
        resultado = modbus_parser.process_single_trace(raw_bytes)
        if resultado is not None:
            resultado['timestamp'] = timestamp
            resultados_validos.append(resultado)
    except Exception:
        pass

if resultados_validos:
    df_seq = pd.DataFrame(resultados_validos)
    df_seq['timestamp'] = pd.to_datetime(df_seq['timestamp'])
    df_seq['delta_t'] = df_seq['timestamp'].diff().dt.total_seconds().fillna(0.0)

end_seq = time.perf_counter()
time_seq = end_seq - start_seq

# ---------------------------------------------------------
# Prueba 2: Paralelo en Batch (Rust + Rayon)
# ---------------------------------------------------------
print("Ejecutando método 2 (Paralelo / Batch)...")
start_par = time.perf_counter()

timestamps_list = df_raw['timestamp'].astype(str).tolist()
hex_list = df_raw['hex_data'].tolist()

resultado_dict = modbus_parser.process_batch_traces(timestamps_list, hex_list)
df_par = pd.DataFrame(resultado_dict)

if not df_par.empty:
    df_par['timestamp'] = pd.to_datetime(df_par['timestamp'])
    df_par['delta_t'] = df_par['timestamp'].diff().dt.total_seconds().fillna(0.0)
    
    # Exportar resultados paralelos si la bandera está activada
    if PROCESS_DATA:
        df_par.to_csv(output_file, index=False)
        print(f"\n[+] Resultados exportados exitosamente a: {output_file}")

end_par = time.perf_counter()
time_par = end_par - start_par

print(f"\nTiempo Secuencial : {time_seq:.4f} segundos")
print(f"Tiempo Paralelo   : {time_par:.4f} segundos")

if time_par > 0:
    print(f"Relacion tiempo secuencial / tiempo paralelo: {time_seq / time_par:.2f}")